In [3]:
import os
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from torch.utils.data import TensorDataset, DataLoader
from torch.serialization import safe_globals

In [ ]:
# Temp
# class SelectorMLP(nn.Module):
#     def __init__(self, input_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(64, 1)
#         )
#     def forward(self, x):
#         return self.net(x).squeeze(-1)

# ---------------------------------------
# Configuration
# ---------------------------------------
parquet_path   = 'Master_cleaned.parquet'
return_col     = 'ret_5d'
batch_size     = 1_000_000   # streaming size for Parquet
test_fraction  = 0.2         # last 20% of dates for test
device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ---------------------------------------
# Build X_test, y_test (same as before)
# ---------------------------------------
pf = pq.ParquetFile(parquet_path)
all_cols = pf.schema_arrow.names
exclude  = {return_col, 'date', 'ticker'}
feature_cols = [c for c in all_cols if c not in exclude]

cutoff_lists, all_dates = {}, set()
for batch in pf.iter_batches(batch_size=batch_size):
    dfb = batch.to_pandas()[['date', return_col]]
    for date, grp in dfb.groupby('date'):
        all_dates.add(date)
        cutoff_lists.setdefault(date, []).extend(grp[return_col].values)
cutoff_map = {d: np.quantile(vals, 0.95) for d, vals in cutoff_lists.items()}
all_dates = sorted(all_dates)

split_idx  = int(len(all_dates) * (1 - test_fraction))
test_dates = set(all_dates[split_idx:])
# split_idx = next(i for i, d in enumerate(all_dates) if str(d) >= '2025-06-15')
# test_dates = set(all_dates[split_idx:])

sample = pf.read_row_group(0).to_pandas()[feature_cols]
scaler = StandardScaler().fit(sample.values)

chunks = []
for batch in pf.iter_batches(batch_size=batch_size):
    df = batch.to_pandas()
    df = df[df['date'].isin(test_dates)]
    if not df.empty:
        chunks.append(df)
test_df = pd.concat(chunks, ignore_index=True)

test_df['label'] = (test_df[return_col] >= test_df['date'].map(cutoff_map)).astype(np.int8)
mask = np.isfinite(test_df[feature_cols].values).all(axis=1)
test_df = test_df.loc[mask].reset_index(drop=True)

X_test = scaler.transform(test_df[feature_cols].values.astype(np.float32))
y_test = test_df['label'].values.astype(np.int8)
print("X_test shape:", X_test.shape, "y_test dist:", np.bincount(y_test))

# ---------------------------------------
# Load the TorchScript model
# ---------------------------------------
with safe_globals([SelectorMLP]):
    scripted_model = torch.load(
        'selector_mlp_model.pt',
        map_location=device,
        weights_only=False
    )
scripted_model.to(device).eval()

# ---------------------------------------
# Batched inference & metrics
# ---------------------------------------
test_ds     = TensorDataset(torch.from_numpy(X_test).float(),
                            torch.from_numpy(y_test).long())
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        logits = scripted_model(Xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(np.int8)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(yb.numpy())

all_probs  = np.concatenate(all_probs)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

auc      = roc_auc_score(all_labels, all_probs)
accuracy = accuracy_score(all_labels, all_preds)

print(f"\nTest AUC:      {auc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n",
      classification_report(all_labels, all_preds, digits=4))


test_df['prob'] = all_probs
test_df['pred'] = (all_probs >= 0.5).astype(np.int8)
label_counts_per_day = test_df.groupby('date')['label'].sum()
print("Average positives per day:", label_counts_per_day.mean())
print("Expected for 5% threshold:", test_df.groupby('date').size().mean() * 0.05)
selected = test_df[test_df['pred'] == 1]
print(len(selected[['date', 'ticker', 'prob']]))

Using device: cpu
X_test shape: (19745256, 35) y_test dist: [18756439   988817]

Test AUC:      0.9964
Test Accuracy: 0.9914

Classification Report:
               precision    recall  f1-score   support

           0     0.9943    0.9966    0.9955  18756439
           1     0.9327    0.8917    0.9118    988817

    accuracy                         0.9914  19745256
   macro avg     0.9635    0.9442    0.9536  19745256
weighted avg     0.9912    0.9914    0.9913  19745256

Average positives per day: 309.39205256570716
Expected for 5% threshold: 308.90575719649564
945405


In [7]:
selected = selected.sort_values(by='prob', ascending=False)
selected

,date,ticker,open,high,low,close,volume,spy_ret,dow_0,dow_1,...,neg_score,neutral_score,pos_score,pos_minus_neg,pos_minus_neg_lag1,headline_count,headline_count_lag1,label,prob,pred
14114265,2023-02-28 00:00:00+00:00,DTSTW,0.31970,0.31970,0.12250,0.19710,1.014300e+04,-0.003036,0,1,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1,1.000000,1
7981557,2019-10-10 00:00:00+00:00,NAT,2.99715,3.33202,2.95533,3.21488,1.217275e+07,0.006416,0,0,...,0.0,0.0,0.0,0.0,0.327883,0.0,6.0,1,1.000000,1
17104007,2024-06-14 00:00:00+00:00,MDIA,2.35000,3.32990,2.35000,2.79000,2.213578e+06,-0.000394,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1,1.000000,1
17948519,2024-10-16 00:00:00+00:00,PHUN,6.21000,6.62000,5.80000,6.34000,9.049393e+06,0.004679,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1,1.000000,1
19010966,2025-03-19 00:00:00+00:00,NIXXW,0.02900,0.03860,0.02410,0.03860,9.277000e+03,0.010799,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1,1.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8914324,2020-05-22 00:00:00+00:00,GMED,57.00000,57.00000,54.20000,55.11000,3.065953e+06,0.002354,0,0,...,0.0,0.0,0.0,0.0,-0.068152,0.0,2.0,1,0.500017,1
8737723,2020-04-13 00:00:00+00:00,CRMT,64.28000,66.54000,63.86000,64.75000,7.833400e+04,-0.010105,1,0,...,0.0,0.0,0.0,0.0,0.236319,0.0,1.0,0,0.500012,1
14108994,2023-02-27 00:00:00+00:00,SKF,50.40280,51.61120,50.25880,51.29520,3.624442e+03,0.003073,1,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1,0.500009,1
2846979,2015-09-01 00:00:00+00:00,PAC,60.94280,62.19590,59.81760,60.51490,2.332409e+05,-0.029576,0,1,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0,0.500005,1
